# Planning e preprocessing di 501, 502 e 503

Questo notebook esegue il passo che sta fra i dataset preparati e il primo training:
`nnUNetv2_plan_and_preprocess` su **Dataset501_WMH**, **Dataset502_ATLAS** e **Dataset503_DBB**,
seguito dalla verifica del piano descritta in [training.md](../docs/training.md) (Step 3 e Step 4).

Non modifica nulla in `nnUNet_raw/`, nulla in `/data01` (sola lettura) e **non tocca gli
`splits_final.json`**: la loro integrita' viene verificata con un hash prima e dopo.

## Che cosa fa davvero il comando

`nnUNetv2_plan_and_preprocess` e' tre operazioni in fila, ed e' utile tenerle distinte perche' solo la
terza costa spazio su disco:

1. **Fingerprint** (`dataset_fingerprint.json`) - legge tutti i volumi di training e ne misura le
   proprieta': spacing (la dimensione fisica di un voxel, in millimetri), shape, distribuzione delle
   intensita' dentro la regione di foreground, quanto foreground c'e' per caso. E' una misura, non una
   scelta.
2. **Planning** (`nnUNetPlans.json`) - da quelle misure deriva le decisioni: a quale griglia comune
   ricampionare tutti i casi (*target spacing*), quanto grande e' la porzione di volume che la rete vede
   in una volta (*patch size*), quanti campioni per passo (*batch size*), profondita' della rete. E' qui
   che nasce l'architettura, ed e' per questo che due dataset che vogliono condividere pesi devono
   condividere il piano.
3. **Preprocessing** - applica il piano ai dati: ricampiona, normalizza, ritaglia la regione non nulla e
   salva tutto in `.npz`/`.b2nd`. E' questa fase che scrive decine di GB.

`--verify_dataset_integrity` viene prima di tutto: controlla che ogni immagine abbia la sua label, che
le geometrie coincidano e che le label dichiarate in `dataset.json` siano quelle presenti nei file. Va
fatto una volta per dataset; se fallisce, il problema e' nella preparazione e non ha senso proseguire.

## Step 0 - Ambiente, GPU e spazio disco

In [1]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time
import pandas as pd

PROJECT = Path('/home/ccantini/lesion_detection_project')
LOG_DIR = PROJECT / 'reports' / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Le variabili nnUNet_* hanno un'unica fonte di verita': scripts/env.sh. Invece di
# riscriverle qui le leggiamo eseguendo lo script, cosi' il notebook non puo'
# divergere dai comandi lanciati a mano dal terminale.
_env = subprocess.run(['bash', '-lc', f'source {PROJECT}/scripts/env.sh && env'],
                      capture_output=True, text=True, check=True).stdout
ENV = dict(os.environ)
for riga in _env.splitlines():
    if riga.startswith('nnUNet_'):
        k, _, v = riga.partition('=')
        ENV[k] = v

RAW = Path(ENV['nnUNet_raw'])
PREP = Path(ENV['nnUNet_preprocessed'])
for k in ('nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results', 'nnUNet_extTrainer'):
    print(f'{k:20s} {ENV[k]}')

nnUNet_raw           /home/ccantini/lesion_detection_project/nnUNet/nnUNet_raw
nnUNet_preprocessed  /home/ccantini/lesion_detection_project/nnUNet/nnUNet_preprocessed
nnUNet_results       /home/ccantini/lesion_detection_project/nnUNet/nnUNet_results
nnUNet_extTrainer    /home/ccantini/lesion_detection_project/scripts/nnunet_trainers


La GPU **non serve** al preprocessing (e' lavoro di CPU e disco): la scegliamo comunque adesso perche'
`CUDA_VISIBLE_DEVICES` deve essere corretta per i training che seguono, e perche' vedere lo stato delle
schede prima di iniziare fa parte della procedura. Le 4 A100 sono condivise con altri utenti, quindi
quale sia libera cambia di giorno in giorno e non si eredita dalla sessione precedente.

In [2]:
q = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,memory.total,utilization.gpu',
                    '--format=csv,noheader,nounits'], capture_output=True, text=True, check=True)
gpu = pd.DataFrame([r.split(', ') for r in q.stdout.strip().splitlines()],
                   columns=['idx', 'mem_usata_MiB', 'mem_tot_MiB', 'util_%']).astype(int)
gpu['libera'] = (gpu.mem_usata_MiB < 5000) & (gpu['util_%'] < 20)
display(gpu)

libere = gpu[gpu.libera]
if libere.empty:
    print('\nATTENZIONE: nessuna GPU risulta libera. Il preprocessing procede lo stesso '
          '(non usa la GPU), ma il training va rimandato.')
    GPU = None
else:
    GPU = int(libere.iloc[0].idx)
    ENV['CUDA_VISIBLE_DEVICES'] = str(GPU)
    print(f'\nGPU scelta per i training successivi: {GPU}')

,idx,mem_usata_MiB,mem_tot_MiB,util_%,libera
0,0,2118,81920,0,True
1,1,53517,81920,99,False
2,2,53529,81920,97,False
3,3,53529,81920,100,False



GPU scelta per i training successivi: 0


In [3]:
SOGLIA_GB = 60.0   # sotto questa soglia non si lancia un altro dataset

def gb_liberi(path='/home'):
    return shutil.disk_usage(path).free / 2**30

def spazio(nota=''):
    """Stampa lo spazio libero su /home e quello occupato dal preprocessed."""
    occupato = sum(f.stat().st_size for f in PREP.rglob('*') if f.is_file()) / 2**30
    print(f'{nota:28s} liberi su /home: {gb_liberi():7.1f} GB   |   nnUNet_preprocessed: {occupato:6.1f} GB')
    return gb_liberi()

def controlla_spazio(prossimo):
    """Guardia: /home e' condiviso ed e' oltre il 90%, meglio fermarsi in modo pulito
    che riempire il disco a meta' di un preprocessing."""
    g = gb_liberi()
    if g < SOGLIA_GB:
        raise RuntimeError(f'Solo {g:.1f} GB liberi (soglia {SOGLIA_GB}): non lancio {prossimo}.')
    print(f'OK: {g:.1f} GB liberi, procedo con {prossimo}.')

spazio('stato iniziale')

stato iniziale               liberi su /home:   250.2 GB   |   nnUNet_preprocessed:    0.2 GB


250.2211151123047

## Step 1 - Stato di partenza e impronta degli split

`splits_final.json` definisce quali casi stanno in training e quali in validation per ciascuno dei 5
fold. Lo abbiamo scritto noi ed e' il riferimento fisso di tutti i confronti: se cambiasse, due run
eseguite prima e dopo non sarebbero piu' paragonabili, perche' la differenza di Dice potrebbe venire
dallo split e non dall'esperimento. nnU-Net lo rigenera **solo se il file non esiste**, quindi
`plan_and_preprocess` non dovrebbe toccarlo — ma invece di fidarci ne prendiamo l'hash adesso e lo
riverifichiamo alla fine.

In [4]:
DATASETS = {501: 'Dataset501_WMH', 502: 'Dataset502_ATLAS', 503: 'Dataset503_DBB'}

# Riferimenti attesi, dai report gia' prodotti (reports/dataset_preparation.md, reports/dbb_analysis.md).
ATTESO = {
    501: dict(spacing=(1.2, 0.9766, 3.0), casi_tr=79,  fonte='reports/dataset_preparation.md'),
    502: dict(spacing=(1.0, 1.0, 1.0),    casi_tr=556, fonte='reports/dataset_preparation.md'),
    503: dict(spacing=(1.0, 1.0, 1.0),    casi_tr=207, fonte='reports/dbb_analysis.md'),
}

def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

righe = []
for d, nome in DATASETS.items():
    dj = json.loads((RAW / nome / 'dataset.json').read_text())
    sp = PREP / nome / 'splits_final.json'
    righe.append(dict(
        dataset=nome,
        canali=', '.join(dj['channel_names'].values()),
        classi=len(dj['labels']) - 1,
        casi_imagesTr=dj['numTraining'],
        attesi=ATTESO[d]['casi_tr'],
        splits_md5=md5(sp)[:12] if sp.exists() else 'ASSENTE',
    ))
STATO_INIZIALE = pd.DataFrame(righe).set_index('dataset')
MD5_PRIMA = dict(zip(STATO_INIZIALE.index, STATO_INIZIALE.splits_md5))
display(STATO_INIZIALE)

,canali,classi,casi_imagesTr,attesi,splits_md5
dataset,,,,,
Dataset501_WMH,FLAIR,1,79,79,e2abaaaaaa19
Dataset502_ATLAS,T1,1,556,556,f475515458ef
Dataset503_DBB,T1,6,207,207,3a8c2034b666


## Step 2 - La funzione che lancia i comandi

Un'unica funzione per tutti e tre i dataset: esegue il comando nell'ambiente caricato da `env.sh`,
stampa l'output riga per riga mentre scorre (cosi' l'avanzamento e' visibile e non si aspetta al buio)
e ne salva una copia in `reports/logs/`. Il parallelismo e' esplicito: `-np 16` per il preprocessing e
`-npfp 16` per il fingerprint. La macchina ha 96 core ed e' scarica, ma e' condivisa, quindi non li
prendiamo tutti. Il numero di processi cambia solo il **tempo**, non il risultato.

In [5]:
NP = 16   # processi per il preprocessing di ciascuna configurazione
NPFP = 16 # processi per l'estrazione del fingerprint

def plan_and_preprocess(d, verifica=True):
    nome = DATASETS[d]
    cmd = ['nnUNetv2_plan_and_preprocess', '-d', str(d), '-npfp', str(NPFP), '-np', str(NP)]
    if verifica:
        cmd.append('--verify_dataset_integrity')

    log = LOG_DIR / f'plan_and_preprocess_{d}.log'
    print('$', ' '.join(cmd), '\n')
    t0 = time.time()
    with open(log, 'w') as fh:
        p = subprocess.Popen(cmd, cwd=PROJECT, env=ENV, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for riga in p.stdout:
            print(riga, end='')
            fh.write(riga)
        rc = p.wait()
    dt = time.time() - t0
    print(f'\n--- {nome}: uscita {rc}, durata {dt/60:.1f} min, log in {log.relative_to(PROJECT)}')
    if rc != 0:
        raise RuntimeError(f'{nome}: plan_and_preprocess fallito (codice {rc})')
    return dt

## Step 3 - Dataset501_WMH (FLAIR, 79 casi)

Il piu' piccolo, e per questo il primo: se qualcosa nella catena e' rotto lo scopriamo in pochi minuti
invece che dopo un'ora di ATLAS. E' anche il caso interessante dal punto di vista della geometria,
perche' le fette sono spesse **3 mm** contro circa 1 mm nel piano: il voxel non e' un cubo ma un
parallelepipedo schiacciato, e questo si ripercuote sul piano che nnU-Net produce.

In [6]:
DURATE = {}
DURATE[501] = plan_and_preprocess(501)

$ nnUNetv2_plan_and_preprocess -d 501 -npfp 16 -np 16 --verify_dataset_integrity 



Fingerprint extraction...
Dataset501_WMH
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer



####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer



Extracting dataset fingerprint:   1%|▏         | 1/79 [00:02<02:52,  2.21s/it]


Extracting dataset fingerprint:  16%|█▋        | 13/79 [00:02<00:07,  9.26it/s]


Extracting dataset fingerprint:  43%|████▎     | 34/79 [00:02<00:01, 30.18it/s]


... (20 outputs left out)

Preprocessing cases:  85%|████████▍ | 67/79 [00:04<00:00, 24.10it/s]


Preprocessing cases: 100%|██████████| 79/79 [00:04<00:00, 16.06it/s]


Configuration: 3d_lowres...
INFO: Configuration 3d_lowres not found in plans file nnUNetPlans.json of dataset Dataset501_WMH. Skipping.



--- Dataset501_WMH: uscita 0, durata 0.4 min, log in reports/logs/plan_and_preprocess_501.log


In [7]:
spazio('dopo 501')
controlla_spazio('Dataset502_ATLAS')

dopo 501                     liberi su /home:   248.7 GB   |   nnUNet_preprocessed:    1.3 GB
OK: 248.7 GB liberi, procedo con Dataset502_ATLAS.


## Step 4 - Dataset502_ATLAS (T1, 556 casi)

Il piu' grande dei tre e quello che consuma piu' disco: e' il passaggio da tenere d'occhio.

In [8]:
DURATE[502] = plan_and_preprocess(502)

$ nnUNetv2_plan_and_preprocess -d 502 -npfp 16 -np 16 --verify_dataset_integrity 



Fingerprint extraction...
Dataset502_ATLAS


Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer



####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################



Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer



Extracting dataset fingerprint:   0%|          | 1/556 [00:02<22:18,  2.41s/it]


... (253 outputs left out)

Preprocessing cases:  98%|█████████▊| 547/556 [00:26<00:00, 22.92it/s]


Preprocessing cases: 100%|██████████| 556/556 [00:27<00:00, 20.39it/s]


Configuration: 3d_lowres...
INFO: Configuration 3d_lowres not found in plans file nnUNetPlans.json of dataset Dataset502_ATLAS. Skipping.



--- Dataset502_ATLAS: uscita 0, durata 1.5 min, log in reports/logs/plan_and_preprocess_502.log


In [9]:
spazio('dopo 502')
controlla_spazio('Dataset503_DBB')

dopo 502                     liberi su /home:   226.4 GB   |   nnUNet_preprocessed:   17.2 GB
OK: 226.4 GB liberi, procedo con Dataset503_DBB.


## Step 5 - Dataset503_DBB (T1, 6 tessuti, 207 casi)

Compito diverso dagli altri due: qui il bersaglio non e' una lesione rara ma la partizione del cervello
in tessuti, con un foreground che occupa circa un voxel su cinque invece di uno su mille. Il fingerprint
era gia' stato estratto quando abbiamo verificato l'integrita' della conversione, quindi quella parte
sara' rapida.

In [10]:
DURATE[503] = plan_and_preprocess(503)

$ nnUNetv2_plan_and_preprocess -d 503 -npfp 16 -np 16 --verify_dataset_integrity 



Fingerprint extraction...
Dataset503_DBB
Using <class 'nnunetv2.imageio.nibabel_reader_writer.NibabelIOWithReorient'> reader/writer


Affine image: [[-9.86663818e-01 -1.59602135e-01 -9.53806564e-03  9.18468628e+01]
 [-1.62543401e-01  9.69076276e-01  3.28213279e-03 -8.88899460e+01]
 [-9.03925952e-03 -4.78869351e-03  9.82091069e-01 -1.57917114e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]] 
Affine seg: [[-9.86663878e-01 -1.59602135e-01 -9.53831896e-03  9.18468628e+01]
 [-1.62543386e-01  9.69076335e-01  3.28211277e-03 -8.88899460e+01]
 [-9.03951656e-03 -4.78871446e-03  9.82091069e-01 -1.57917114e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Image files: ['/home/ccantini/lesion_detection_project/nnUNet/nnUNet_raw/Dataset503_DBB/imagesTr/sub-0829_0000.nii.gz']. 
Seg file: /home/ccantini/lesion_detection_project/nnUNet/nnUNet_raw/Dataset503_DBB/labelsTr/sub-0829.nii.gz.
This is a warning only and will not cause a crash. Please verify that your segmentation and image are correctly aligned by running: nnUNetv2_plot_overlay_pngs


####################
verify_dataset_integri

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################



Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [182. 218. 166.], 3d_lowres: [182, 218, 166]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 75, 'patch_size': (np.int64(224), np.int64(192)), 'median_image_size_in_voxels': array([218., 166.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynam

Preprocessing cases:   0%|          | 1/207 [00:03<11:21,  3.31s/it]


... (91 outputs left out)

Preprocessing cases:  96%|█████████▌| 198/207 [00:21<00:00, 13.38it/s]


Preprocessing cases:  99%|█████████▊| 204/207 [00:21<00:00, 15.77it/s]


Preprocessing cases: 100%|██████████| 207/207 [00:22<00:00,  9.29it/s]
Configuration: 3d_lowres...
INFO: Configuration 3d_lowres not found in plans file nnUNetPlans.json of dataset Dataset503_DBB. Skipping.



--- Dataset503_DBB: uscita 0, durata 1.0 min, log in reports/logs/plan_and_preprocess_503.log


In [11]:
spazio('dopo 503')

dopo 503                     liberi su /home:   216.3 GB   |   nnUNet_preprocessed:   24.0 GB


216.29954147338867

## Step 6 - Verifica che gli split non siano cambiati

In [12]:
ok = True
for d, nome in DATASETS.items():
    dopo = md5(PREP / nome / 'splits_final.json')[:12]
    uguale = (dopo == MD5_PRIMA[nome])
    ok &= uguale
    print(f'{nome:20s} md5 prima {MD5_PRIMA[nome]}  dopo {dopo}  {"invariato" if uguale else "CAMBIATO!"}')
print('\nTutti gli split sono invariati: i confronti fra run restano validi.' if ok
      else '\nATTENZIONE: uno split e cambiato, i confronti con le run precedenti non sono piu validi.')

Dataset501_WMH       md5 prima e2abaaaaaa19  dopo e2abaaaaaa19  invariato
Dataset502_ATLAS     md5 prima f475515458ef  dopo f475515458ef  invariato
Dataset503_DBB       md5 prima 3a8c2034b666  dopo 3a8c2034b666  invariato

Tutti gli split sono invariati: i confronti fra run restano validi.


Controlliamo anche che le dimensioni dei fold siano quelle attese (Step 4 di `training.md`). nnU-Net le
ristampa all'avvio del training, ma leggerle adesso dal file evita di scoprire un problema dopo aver
occupato la GPU.

In [13]:
ATTESI_FOLD = {
    'Dataset501_WMH':   ['63/16', '63/16', '63/16', '63/16', '64/15'],
    'Dataset502_ATLAS': ['444/112 o 445/111'] * 5,
    'Dataset503_DBB':   ['184/23', '166/41', '166/41', '166/41', '166/41'],
}
righe = []
for nome in DATASETS.values():
    s = json.loads((PREP / nome / 'splits_final.json').read_text())
    righe.append(dict(dataset=nome,
                      fold=[f'{len(f["train"])}/{len(f["val"])}' for f in s],
                      atteso=ATTESI_FOLD[nome]))
display(pd.DataFrame(righe).set_index('dataset'))

,fold,atteso
dataset,,
Dataset501_WMH,"[63/16, 63/16, 63/16, 63/16, 64/15]","[63/16, 63/16, 63/16, 63/16, 64/15]"
Dataset502_ATLAS,"[444/112, 445/111, 445/111, 445/111, 445/111]","[444/112 o 445/111, 444/112 o 445/111, 444/112..."
Dataset503_DBB,"[184/23, 166/41, 166/41, 166/41, 166/41]","[184/23, 166/41, 166/41, 166/41, 166/41]"


## Step 7 - Verifica del piano: la target spacing e' quella prevista?

E' il controllo che rende interpretabili tutti i numeri che verranno dopo, ed e' quello che si salta piu'
spesso. La domanda: la griglia comune che nnU-Net ha scelto coincide con quella che avevamo calcolato noi
misurando i dati? Se no, il fingerprint ha visto dati diversi da quelli attesi — per esempio casi che non
avrebbero dovuto entrare in `imagesTr`.

**Un dettaglio che confonde:** i valori nel piano non sono nell'ordine `x, y, z` dei nostri report.
nnU-Net legge i volumi in ordine inverso e in piu' applica una permutazione degli assi
(`transpose_forward`), scelta per mettere per prima la direzione a risoluzione peggiore. Quindi la stessa
spacing puo' comparire come `[3.0, 1.2, 0.9766]` invece di `1.2 x 0.9766 x 3.0`. Sono la stessa cosa: qui
confrontiamo i valori **ordinati**, che sono invarianti alla permutazione, e mostriamo entrambe le forme.

In [14]:
righe = []
for d, nome in DATASETS.items():
    piano = json.loads((PREP / nome / 'nnUNetPlans.json').read_text())
    atteso = sorted(ATTESO[d]['spacing'])
    for cfg, c in piano['configurations'].items():
        if 'spacing' not in c:
            continue
        sp = [round(float(v), 4) for v in c['spacing']]
        # la 2d ha spacing a 2 elementi (nel piano di fetta): niente confronto 3D
        coincide = (sorted(sp) == [round(v, 4) for v in atteso]) if len(sp) == 3 else '-'
        righe.append(dict(
            dataset=nome, configurazione=cfg,
            spacing_piano=sp,
            spacing_attesa=list(ATTESO[d]['spacing']),
            coincide=coincide,
            patch_size=c.get('patch_size'),
            batch_size=c.get('batch_size'),
            stadi=len(c.get('architecture', {}).get('arch_kwargs', {}).get('strides', [])) or None,
        ))
PIANI = pd.DataFrame(righe).set_index(['dataset', 'configurazione'])
display(PIANI)

for d, nome in DATASETS.items():
    piano = json.loads((PREP / nome / 'nnUNetPlans.json').read_text())
    print(f'{nome:20s} transpose_forward {piano["transpose_forward"]}  '
          f'normalizzazione {piano["configurations"]["3d_fullres"]["normalization_schemes"]}  '
          f'configurazioni pianificate: {list(piano["configurations"])}')

spacing_piano      spacing_attesa  \
dataset          configurazione                                           
Dataset501_WMH   2d                   [0.9766, 1.2]  [1.2, 0.9766, 3.0]   
                 3d_fullres      [3.0, 0.9766, 1.2]  [1.2, 0.9766, 3.0]   
Dataset502_ATLAS 2d                      [1.0, 1.0]     [1.0, 1.0, 1.0]   
                 3d_fullres         [1.0, 1.0, 1.0]     [1.0, 1.0, 1.0]   
Dataset503_DBB   2d                      [1.0, 1.0]     [1.0, 1.0, 1.0]   
                 3d_fullres         [1.0, 1.0, 1.0]     [1.0, 1.0, 1.0]   

                                coincide       patch_size  batch_size  stadi  
dataset          configurazione                                               
Dataset501_WMH   2d                    -       [256, 160]          80      6  
                 3d_fullres         True   [64, 224, 128]           2      6  
Dataset502_ATLAS 2d                    -       [256, 224]          57      6  
                 3d_fullres         True  [128, 128, 128]           2      6  
Dataset503_DBB   2d                    -       [224, 192]          75      6  
                 3d_fullres         True  [128, 160, 112]           2      6

Dataset501_WMH       transpose_forward [0, 1, 2]  normalizzazione ['ZScoreNormalization']  configurazioni pianificate: ['2d', '3d_fullres']
Dataset502_ATLAS     transpose_forward [0, 1, 2]  normalizzazione ['ZScoreNormalization']  configurazioni pianificate: ['2d', '3d_fullres']
Dataset503_DBB       transpose_forward [0, 1, 2]  normalizzazione ['ZScoreNormalization']  configurazioni pianificate: ['2d', '3d_fullres']


Due letture aggiuntive sulla tabella, entrambe indicate in `training.md`:

- **patch size lungo l'asse a 3 mm di WMH.** I volumi hanno circa 83 fette. Se la patch ne copre poche, la
  rete 3D vede pochissimo contesto attraverso il piano, ed e' l'argomento concreto a favore della
  configurazione `2d` — che su questo dataset e' una candidata seria e non una formalita'.
- **batch size.** Il planner lo sceglie in base alla memoria GPU stimata; se e' 2, cioe' il minimo
  ammesso, il modello sta gia' saturando il budget di memoria previsto e la patch e' grande quanto puo'.

In [15]:
p501 = json.loads((PREP / 'Dataset501_WMH' / 'nnUNetPlans.json').read_text())
fp = json.loads((PREP / 'Dataset501_WMH' / 'dataset_fingerprint.json').read_text())
c3d = p501['configurations']['3d_fullres']
tf = p501['transpose_forward']
asse_spesso = int(max(range(3), key=lambda i: c3d['spacing'][i]))
import numpy as np
shape_mediana = np.median(np.array(fp['shapes_after_crop']), axis=0)[tf]
print(f"WMH 3d_fullres: spacing {c3d['spacing']}, patch {c3d['patch_size']}")
print(f"  asse a risoluzione peggiore: indice {asse_spesso} ({c3d['spacing'][asse_spesso]} mm)")
print(f"  la patch ne copre {c3d['patch_size'][asse_spesso]} fette su una mediana di "
      f"{shape_mediana[asse_spesso]:.0f} dopo il crop "
      f"({100 * c3d['patch_size'][asse_spesso] / shape_mediana[asse_spesso]:.0f}% del volume)")
print(f"  batch size {c3d['batch_size']}" + ('  <- minimo: memoria satura' if c3d['batch_size'] == 2 else ''))

WMH 3d_fullres: spacing [3.0, 0.9765999913215637, 1.2000000476837158], patch [64, 224, 128]
  asse a risoluzione peggiore: indice 0 (3.0 mm)
  la patch ne copre 64 fette su una mediana di 79 dopo il crop (81% del volume)
  batch size 2  <- minimo: memoria satura


## Step 8 - Riepilogo

In [16]:
righe = []
for d, nome in DATASETS.items():
    cart = PREP / nome
    dim = {}
    for sub in cart.iterdir():
        if sub.is_dir():
            dim[sub.name] = sum(f.stat().st_size for f in sub.rglob('*') if f.is_file()) / 2**30
    righe.append(dict(dataset=nome,
                      durata_min=round(DURATE.get(d, float('nan')) / 60, 1),
                      totale_GB=round(sum(dim.values()), 1),
                      per_configurazione={k: round(v, 1) for k, v in sorted(dim.items())}))
display(pd.DataFrame(righe).set_index('dataset'))
spazio('finale')
print('\nProssimo passo: fold 0 di 3d_fullres su 501, 502 e 503 con -tr nnUNetTrainerWandb.')

,durata_min,totale_GB,per_configurazione
dataset,,,
Dataset501_WMH,0.4,1.1,"{'gt_segmentations': 0.0, 'nnUNetPlans_2d': 0...."
Dataset502_ATLAS,1.5,15.9,"{'gt_segmentations': 0.0, 'nnUNetPlans_2d': 8...."
Dataset503_DBB,1.0,6.8,"{'gt_segmentations': 0.1, 'nnUNetPlans_2d': 3...."


finale                       liberi su /home:   216.3 GB   |   nnUNet_preprocessed:   24.0 GB

Prossimo passo: fold 0 di 3d_fullres su 501, 502 e 503 con -tr nnUNetTrainerWandb.
